#  Bogor Sport & Tourism Web Scraper

Scraper dengan **kolom terpisah** untuk setiap informasi

In [2]:
import pandas as pd

In [ ]:
!pip install requests beautifulsoup4 pandas openpyxl lxml -q


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict
from IPython.display import display

# PERBAIKAN: senibudaya tanpa dash
CATEGORIES = ["arena", "olahraga", "alam", "senibudaya", "belanja", "kuliner", "rekreasi"]
BASE_URL = "https://sportandtourism.bogorkab.go.id"

print(" Libraries imported!")
print(f" Kategori: {CATEGORIES}")

 Libraries imported!
 Kategori: ['arena', 'olahraga', 'alam', 'senibudaya', 'belanja', 'kuliner', 'rekreasi']


In [ ]:
@dataclass
class TourismItem:
    nama: str
    kategori: str
    label: str
    author: str
    likes: int
    url: str
    url_gambar: str
    deskripsi: str = ""
    alamat: str = ""
    fasilitas: str = ""
    harga_tiket: str = ""
    jam_operasional: str = ""
    telepon: str = ""
    sumber: str = ""
    tags: str = ""

print(" Data structure ready!")

 Data structure ready!


In [ ]:
class BogorTourismScraper:
    def __init__(self, delay: float = 1.0, scrape_details: bool = True):
        self.delay = delay
        self.scrape_details = scrape_details
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        })
    
    def _get_page(self, url: str) -> Optional[BeautifulSoup]:
        try:
            response = self.session.get(url, timeout=30)
            response.raise_for_status()
            return BeautifulSoup(response.text, 'html.parser')
        except Exception as e:
            print(f" Error fetching {url}: {e}")
            return None
    
    def _find_section_content(self, article, keywords: List[str]) -> str:
        """FIXED: Cari konten berdasarkan pattern <p><strong>Keyword</strong></p> lalu ambil sibling berikutnya"""
        for kw in keywords:
            # Cari <strong> yang mengandung keyword
            for strong in article.find_all('strong'):
                if kw.lower() in strong.get_text(strip=True).lower():
                    # Cari parent <p>
                    parent_p = strong.find_parent('p')
                    if parent_p:
                        # Ambil sibling berikutnya
                        next_elem = parent_p.find_next_sibling()
                        if next_elem:
                            # Jika <ul>, ambil semua <li>
                            if next_elem.name == 'ul':
                                items = [li.get_text(strip=True) for li in next_elem.find_all('li')]
                                return ' | '.join(items)
                            # Jika <p>, ambil teks
                            elif next_elem.name == 'p':
                                return next_elem.get_text(strip=True)
                            # Jika lainnya
                            else:
                                return next_elem.get_text(strip=True)
        return ""
    
    def _get_detail_data(self, url: str) -> Dict[str, str]:
        """FIXED: Extract detail dari halaman dengan struktur yang benar"""
        result = {'deskripsi': '', 'alamat': '', 'fasilitas': '', 'harga_tiket': '', 
                  'jam_operasional': '', 'telepon': '', 'sumber': '', 'tags': ''}
        if not url:
            return result
        
        try:
            soup = self._get_page(url)
            if not soup:
                return result
            
            # PENTING: Gunakan <article> sebagai container untuk menghindari tags/categories
            article = soup.find('article')
            if not article:
                article = soup.find('div', class_='wrap-fullwidth') or soup.find('div', class_='entry-content')
            if not article:
                return result
            
            # Hapus elemen yang tidak diperlukan
            for unwanted in article.find_all(['form', 'footer', 'nav']):
                unwanted.decompose()
            for unwanted in article.find_all(class_=re.compile(r'comment|reply|related|share|social|article-category|article-tag')):
                unwanted.decompose()
            
            # Deskripsi: paragraf awal sebelum section detail
            paragraphs = []
            stop_keywords = ['alamat', 'fasilitas', 'harga', 'tiket', 'jam operasional', 'telepon', 'lokasi']
            for p in article.find_all('p', recursive=True):
                txt = p.get_text(strip=True)
                # Jika ini adalah header section, berhenti
                strong = p.find('strong')
                if strong and any(kw in strong.get_text(strip=True).lower() for kw in stop_keywords):
                    break
                if txt and len(txt) > 30 and not any(kw in txt.lower()[:50] for kw in stop_keywords):
                    paragraphs.append(txt)
            result['deskripsi'] = '\n\n'.join(paragraphs[:5])
            
            # FIXED: Gunakan _find_section_content untuk extract data dengan benar
            result['alamat'] = self._find_section_content(article, ['Alamat', 'Lokasi'])
            result['fasilitas'] = self._find_section_content(article, ['Fasilitas'])
            result['harga_tiket'] = self._find_section_content(article, ['Harga Tiket', 'HTM', 'Tiket Masuk', 'Harga'])
            result['jam_operasional'] = self._find_section_content(article, ['Jam Operasional', 'Jam Buka', 'Waktu Operasional'])
            result['telepon'] = self._find_section_content(article, ['Telepon', 'Telp', 'No. Telp', 'Kontak'])
            
            # Fallback telepon: cari pattern nomor telepon
            if not result['telepon']:
                text = article.get_text()
                match = re.search(r'(\+62|62|08|021)[\d\-\s]{8,15}', text)
                if match:
                    result['telepon'] = match.group(0).strip()
            
            # Fallback alamat: cari pattern alamat
            if not result['alamat']:
                for p in article.find_all('p'):
                    txt = p.get_text(strip=True)
                    if any(kw in txt.lower() for kw in ['jl.', 'jalan', 'kecamatan', 'kab.', 'kabupaten', 'desa', 'kp.', 'kampung']):
                        if len(txt) > 20 and len(txt) < 300:
                            result['alamat'] = txt
                            break
            
            # Sumber: link eksternal
            for a in article.find_all('a', href=True):
                href = a.get('href', '')
                if any(x in href for x in ['travelspromo', 'google.com/maps', 'goo.gl', 'maps.google']):
                    result['sumber'] = href
                    break
            
            # Tags: dari div.article-tags (di luar article)
            tags_div = soup.find('div', class_='article-tags') or soup.find(class_=re.compile(r'tags'))
            if tags_div:
                result['tags'] = ', '.join([a.get_text(strip=True) for a in tags_div.find_all('a')])
            
            return result
        except Exception as e:
            print(f" Error detail: {e}")
            return result
    
    def _extract_items(self, soup: BeautifulSoup, kategori: str) -> List[TourismItem]:
        items = []
        container = soup.find('ul', id='infinite-articles') or soup.find('ul', class_='masonry_list')
        if not container:
            print(f"    Container tidak ditemukan")
            return []
        
        list_items = container.find_all('li', class_=lambda x: x and 'post' in str(x))
        print(f"    Ditemukan {len(list_items)} items")
        
        for item in list_items:
            try:
                nama = ""
                content_div = item.find('div', class_='content-masonry')
                if content_div:
                    h3 = content_div.find('h3')
                    if h3:
                        nama = h3.get_text(strip=True)
                if not nama:
                    for h3 in item.find_all('h3'):
                        if 'index-title' not in str(h3.get('class', [])):
                            nama = h3.get_text(strip=True)
                            break
                
                url = ""
                a = item.find('a', href=True)
                if a:
                    url = a.get('href', '')
                    if url and not url.startswith('http'):
                        url = BASE_URL + url
                
                label = ""
                cat_div = item.find('div', class_='article-category')
                if cat_div:
                    cat_a = cat_div.find('a')
                    if cat_a:
                        label = cat_a.get_text(strip=True)
                
                author = ""
                meta = item.find('ul', class_='meta-content')
                if meta:
                    auth = meta.find('a')
                    if auth:
                        author = auth.get_text(strip=True)
                
                likes = 0
                lk = item.find('span', class_='thumbs-rating-up')
                if lk:
                    nums = re.findall(r'\d+', lk.get_text(strip=True))
                    if nums:
                        likes = int(nums[0])
                
                url_gambar = ""
                img = item.find('img')
                if img:
                    url_gambar = img.get('src') or img.get('data-src') or ""
                
                if nama:
                    items.append(TourismItem(
                        nama=nama, kategori=kategori.title(),
                        label=label, author=author, likes=likes, url=url, url_gambar=url_gambar
                    ))
            except Exception as e:
                continue
        return items
    
    def _get_max_pages(self, soup) -> int:
        mx = 1
        for a in soup.find_all('a', href=re.compile(r'/page/\d+')):
            m = re.search(r'/page/(\d+)', a.get('href', ''))
            if m:
                mx = max(mx, int(m.group(1)))
        return mx
    
    def scrape_category(self, kategori: str, max_pages: int = None) -> List[TourismItem]:
        k = kategori.lower().replace(' ', '')
        if k not in CATEGORIES:
            print(f" Kategori '{kategori}' tidak valid")
            return []
        
        all_items = []
        print(f"\n Scraping: {kategori}")
        
        soup = self._get_page(f"{BASE_URL}/category/{k}/")
        if not soup:
            return []
        
        mx = min(self._get_max_pages(soup), max_pages or 999)
        print(f"    Total halaman: {mx}")
        
        items = self._extract_items(soup, kategori)
        all_items.extend(items)
        print(f"    Page 1: {len(items)} items")
        
        for pg in range(2, mx + 1):
            time.sleep(self.delay)
            soup = self._get_page(f"{BASE_URL}/category/{k}/page/{pg}/")
            if not soup:
                break
            items = self._extract_items(soup, kategori)
            if not items:
                break
            all_items.extend(items)
            print(f"    Page {pg}: {len(items)} items")
        
        if self.scrape_details and all_items:
            print(f"    Mengambil detail...")
            for i, it in enumerate(all_items):
                time.sleep(self.delay * 0.5)
                d = self._get_detail_data(it.url)
                it.deskripsi = d['deskripsi']
                it.alamat = d['alamat']
                it.fasilitas = d['fasilitas']
                it.harga_tiket = d['harga_tiket']
                it.jam_operasional = d['jam_operasional']
                it.telepon = d['telepon']
                it.sumber = d['sumber']
                it.tags = d['tags']
                if (i+1) % 5 == 0:
                    print(f"      {i+1}/{len(all_items)}")
        
        print(f"    Total: {len(all_items)}")
        return all_items
    
    def scrape_all(self, max_pages_per_cat: int = None) -> List[TourismItem]:
        all_items = []
        print("="*50)
        print(" BOGOR TOURISM SCRAPER")
        print("="*50)
        for k in CATEGORIES:
            all_items.extend(self.scrape_category(k, max_pages_per_cat))
            time.sleep(self.delay)
        print(f"\n TOTAL: {len(all_items)} destinasi")
        return all_items
    
    def to_dataframe(self, items) -> pd.DataFrame:
        return pd.DataFrame([asdict(i) for i in items])

print(" Scraper ready!")

 Scraper ready!


---
##  Test Scrape Satu Kategori

In [ ]:
scraper = BogorTourismScraper(delay=1.5, scrape_details=True)

# Test satu kategori, max 1 halaman
items = scraper.scrape_category("alam", max_pages=1)

if items:
    df = scraper.to_dataframe(items)
    print(f"\n Jumlah data: {len(df)}")
    print(f" Kolom: {df.columns.tolist()}")
    display(df[['nama', 'alamat', 'fasilitas', 'harga_tiket', 'jam_operasional']].head(10))
else:
    print(" Tidak ada data!")


 Scraping: alam
    Total halaman: 1
    Ditemukan 10 items
    Page 1: 10 items
    Mengambil detail...
      5/10
      10/10
    Total: 10

 Jumlah data: 10
 Kolom: ['nama', 'kategori', 'label', 'author', 'likes', 'url', 'url_gambar', 'deskripsi', 'alamat', 'fasilitas', 'harga_tiket', 'jam_operasional', 'telepon', 'sumber', 'tags']


,nama,alamat,fasilitas,harga_tiket,jam_operasional
0,Curug Kiara,"Kampung Raina, Desa Ciasihan, Kecamatan Pamija...",Di area Curug Kiara belum ada fasilitas apa-ap...,Di awal masuk kamu harus membayar sebesar Rp. ...,24 jam
1,Leuwi Pangaduan,"Jl. Kp. Muhara No.Ds, Bojong Koneng, Kecamatan...",Area parkir | Toilet | Kamar mandi | Mushola |...,Dewasa Rp.30.000 | Anak-anak Rp.20.000 | Parki...,Setiap hari Jam 08.00-17.000
2,Curug Ciampea,"Curug Ciampea Bogor berada di Tapos I, Kecamat...",Tempat Parkir | Kamar Mandi | camp ground,Rp.22.000 untuk masuk | Rp.30.000 untuk camp,Curug Ciampea Bogor buka dari hari Senin sampa...
3,Curug Cikoneng,Lokasi dan alamat Curug Cikoneng terletak di K...,Area parkir motor yang cukup luas | Kamar mand...,Tiket masuk memasuki area Curug Cikoneng sanga...,dibuka selama 24 jam sehari dan 7 hari seminggu.
4,Curug Walet,"Desa Ciasihan, Kecamatan Pamijahan, Kabupten B...",Area parkir | Warung | Kamar mandi,Tiket Masuk : 10.000 | Parkir : 5.000,Rabu 07.00–16.00 | Kamis 07.00–05.00 | Jumat 0...
5,Curug Rahong,Objek wisata air terjun ini berada di Kampung ...,"toilet, kamar bilas, serta warung makan dengan...",Rp.10.000,Setiap hari 07:00-17:30
6,Bukit Cirimpak,Lokasi Bukit Cirimpak berada di Jl. Curug Panj...,Toilet | Warung makanan dan minuman | Camping ...,"Tiket masuk Bukit Cirimpak sebesar Rp. 35.000,...",Jam buka Bukit Cirimpak setiap hari Senin hing...
7,Curug Bungsu,Curug ini berada di daerah hutan lindung Gunun...,Area parkirKamar mandiToilet,Rp. 10.000 setiap orang.,"Tidak diketahui, bisa langsung mengunjunginya ..."
8,Lembah Tepus,Lembah Tepus adalah bagian dari Taman Nasional...,Lembah Tepus memiliki area parkir yang luas. W...,Masuk wisata 10.000Masuk Camping 30.000,Setiap Hari 07:00-17:00Camping 24jam
9,Curug Cikawah,Lokasi dan Alamat Curug Cikawah berada di Desa...,Fasilitas yang ada disekitar Curug Cikawah tid...,Tiket masuk Curug Cikawah cukup murah yakni ka...,Jam operasional Curug Cikawah dibuka selama 7 ...


In [ ]:
# Lihat detail satu destinasi
if items:
    row = df.iloc[0]
    print(f" {row['nama']}")
    print(f"\n Deskripsi:\n{row['deskripsi'][:300]}..." if len(str(row['deskripsi'])) > 300 else f"\n Deskripsi:\n{row['deskripsi']}")
    print(f"\n Alamat: {row['alamat']}")
    print(f" Fasilitas: {row['fasilitas']}")
    print(f" Harga Tiket: {row['harga_tiket']}")
    print(f" Jam Operasional: {row['jam_operasional']}")
    print(f" Telepon: {row['telepon']}")

 Curug Kiara

 Deskripsi:
Namanya sangat indah sesuai dengan keindahan yang akan kamu dapatkan ketika berkunjung ke Curug Kiara. Berikut ulasan lengkap mengenai Curug Kiara, check this out!

Perjalanan menuju Curug Kiara sungguh sangat mengagumkan, kamu dapat melihat hamparan pemandangan yang menyejukkan.

Jalan yang akan di...

 Alamat: Kampung Raina, Desa Ciasihan, Kecamatan Pamijahan,Kab Bogor, Jawa Barat.
 Fasilitas: Di area Curug Kiara belum ada fasilitas apa-apa karena lokasinya yang berada di tengah hutan. Namun di sekitar Curug Kiara terdapat beberapa warung makanan dan minuman, saung sederhana untuk kamu beristirahat sejenak. Serta terdapat area parkir yang cukup untuk menampung beberapa motor.
 Harga Tiket: Di awal masuk kamu harus membayar sebesar Rp. 25.000,- per motor dua orang. Atau Rp. 10.000,- per orang dan untuk parkir mobil Rp. 10.000,-
 Jam Operasional: 24 jam
 Telepon: 


##  Scrape Semua Kategori

In [ ]:
scraper = BogorTourismScraper(delay=1.5, scrape_details=True)
all_items = scraper.scrape_all(max_pages_per_cat=None)
df_all = scraper.to_dataframe(all_items)
print(f"\n Total: {len(df_all)} destinasi")

 BOGOR TOURISM SCRAPER

 Scraping: arena
    Total halaman: 9
    Ditemukan 10 items
    Page 1: 10 items
    Ditemukan 10 items
    Page 2: 10 items
    Ditemukan 10 items
    Page 3: 10 items
    Ditemukan 10 items
    Page 4: 10 items
    Ditemukan 10 items
    Page 5: 10 items
    Ditemukan 10 items
    Page 6: 10 items
    Ditemukan 10 items
    Page 7: 10 items
    Ditemukan 10 items
    Page 8: 10 items
    Ditemukan 3 items
    Page 9: 3 items
    Mengambil detail...
      5/83
      10/83
      15/83
      20/83
      25/83
      30/83
      35/83
      40/83
      45/83
      50/83
      55/83
      60/83
      65/83
      70/83
      75/83
      80/83
    Total: 83

 Scraping: olahraga
    Total halaman: 2
    Ditemukan 10 items
    Page 1: 10 items
    Ditemukan 10 items
    Page 2: 10 items
    Mengambil detail...
      5/20
      10/20
      15/20
      20/20
    Total: 20

 Scraping: alam
    Total halaman: 13
    Ditemukan 10 items
    Page 1: 10 items
    Ditemukan 10 

In [ ]:
display(df_all[['nama', 'kategori', 'alamat', 'fasilitas', 'harga_tiket', 'jam_operasional']].head(20))

,nama,kategori,alamat,fasilitas,harga_tiket,jam_operasional
0,Curug Ciampea,Arena,"Curug Ciampea Bogor berada di Tapos I, Kecamat...",Tempat Parkir | Kamar Mandi | camp ground,Rp.22.000 untuk masuk | Rp.30.000 untuk camp,Curug Ciampea Bogor buka dari hari Senin sampa...
1,Bukit Cirimpak,Arena,Lokasi Bukit Cirimpak berada di Jl. Curug Panj...,Toilet | Warung makanan dan minuman | Camping ...,"Tiket masuk Bukit Cirimpak sebesar Rp. 35.000,...",Jam buka Bukit Cirimpak setiap hari Senin hing...
2,Lembah Tepus,Arena,Lembah Tepus adalah bagian dari Taman Nasional...,Lembah Tepus memiliki area parkir yang luas. W...,Masuk wisata 10.000Masuk Camping 30.000,Setiap Hari 07:00-17:00Camping 24jam
3,Sun Water Park Kahuripan,Arena,Sun Water Park berlokasi di Jalan Kampung Dure...,"toilet, tempat makan, parkir, musala, dan ruan...",,
4,Lembah Pinus Camp & Café,Arena,"9266+4C9, Jl. Puncak Dua, Wargajaya, Kec. Suka...","Area camp-nya terbagi atas blok jambore, blok ...",Biaya sewa tenda di Lembah Pinus Camp & Cafe m...,Buka 24 jam
5,Bukit Alas Bandawasa,Arena,Area bumi perkemahan ini terletak di kaki Gunu...,Fasilitas yang tersedia di bumi perkemahan Buk...,,Selalu buka 24 jam
6,Nirwana Golden Park,Arena,"GR6J+7R3, Jl. Nirwana Golden Park, Pakansari, ...",,Harga Tiket Kolam Renang Nirwana Golden ParkWe...,08.00-18.00
7,Rumah Ibu Waterboom,Arena,"Jl. Tegar Beriman, Bojong Baru, Kecamatan Bojo...",Tempat parkir yang luas | Kolam ada 4 | Peroso...,Weekday 35.000 | Weekend 50.000,08.00-17.00
8,Gumati Waterpark,Arena,"Jl. Babakan Tumas No.16, Cikeas, Kec. Sukaraja...",Sarana yang ada untuk mendukung service pengun...,Tiket Masuk WeekdayRp40.000Tiket Masuk Weekend...,Weekday 08.00-17.00
9,Boash Waterpark,Arena,Kolam renang dan waterpak ini berada di Desa B...,Toilet | kamar ganti | tempat bilas | loker | ...,,Selasa08.00–16.00Rabu08.00–16.00Kamis08.00–16....


##  Simpan Data

In [13]:
import pandas as pd
from pathlib import Path

# Jika df_all belum ada di kernel, pakai data clean yang sudah tersimpan.
# Saat cell scraping utama dijalankan dulu, df_all tetap berisi raw 430 dari hasil scraping.
if "df_all" not in globals():
    clean_path = Path("bogor_tourism_data_clean.csv")
    if not clean_path.exists():
        clean_path = Path("../script/bogor_tourism_data_clean.csv")
    df_all = pd.read_csv(clean_path)

df_raw = df_all.copy()

for _col in df_raw.select_dtypes(include=["object"]).columns:
    df_raw[_col] = df_raw[_col].map(
        lambda value: "
".join(line.rstrip() for line in value.splitlines())
        if isinstance(value, str)
        else value
    )

# Versi clean: data unik berdasarkan URL dan nama destinasi.
# Jika nama destinasi sama, simpan baris dengan likes paling tinggi.
df_unique = df_raw.drop_duplicates(subset="url").copy()
df_unique["_row_order"] = range(len(df_unique))
df_unique["_nama_norm"] = (
    df_unique["nama"].astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.strip()
)
df_unique["_likes_sort"] = pd.to_numeric(df_unique["likes"], errors="coerce").fillna(0)
df_unique = (
    df_unique.sort_values(["_nama_norm", "_likes_sort", "_row_order"], ascending=[True, False, True])
    .drop_duplicates(subset="_nama_norm", keep="first")
    .sort_values("_row_order")
    .drop(columns=["_row_order", "_nama_norm", "_likes_sort"])
    .reset_index(drop=True)
)

print(f"Data sumber: {len(df_raw)} destinasi")
print(df_raw["kategori"].value_counts())

print("
Clean:", len(df_unique), "destinasi")
print(df_unique["kategori"].value_counts())


Data sumber: 430 destinasi
kategori
Rekreasi       135
Alam           122
Arena           83
Kuliner         41
Olahraga        20
Seni Budaya     19
Belanja         10
Name: count, dtype: int64

Clean: 296 destinasi
kategori
Arena          82
Alam           82
Rekreasi       62
Kuliner        38
Olahraga       11
Seni Budaya    11
Belanja        10
Name: count, dtype: int64


In [14]:
# Raw 430 tidak disimpan agar folder script tetap bersih.
print("Raw 430 tidak disimpan")


Raw 430 tidak disimpan


In [15]:
import pandas as pd
from pathlib import Path

if "df_unique" not in globals():
    clean_path = Path("bogor_tourism_data_clean.csv")
    if not clean_path.exists():
        clean_path = Path("../script/bogor_tourism_data_clean.csv")
    df_unique = pd.read_csv(clean_path)

# Simpan hanya versi clean yang dipakai project.
df_unique.to_csv("bogor_tourism_data_clean.csv", index=False, encoding="utf-8-sig")
df_unique.to_json("bogor_tourism_data_clean.json", orient="records", force_ascii=False, indent=2)
df_unique.to_excel("bogor_tourism_data_clean.xlsx", index=False, engine="openpyxl")

print("Saved clean 296")


Saved clean 296
